# AENZO LABS — Telegram Session Login\nRun the next cell. Enter credentials only in the private widget fields. Do not paste the session string into chat or commit it to GitHub.

In [ ]:
!pip -q install telethon ipywidgets\nfrom google.colab import output\noutput.enable_custom_widget_manager()\nimport asyncio
import ipywidgets as widgets
from IPython.display import display
from telethon import TelegramClient
from telethon.sessions import StringSession
from telethon.errors import SessionPasswordNeededError

api_id_box = widgets.Text(description="API ID:", placeholder="Telegram API ID")
api_hash_box = widgets.Password(description="API Hash:", placeholder="Paste privately")
phone_box = widgets.Text(description="Phone:", placeholder="+countrycode number")
start_button = widgets.Button(description="Send login code", button_style="primary")
code_box = widgets.Text(description="Login code:", placeholder="Code from Telegram")
verify_button = widgets.Button(description="Verify code", button_style="success")
password_box = widgets.Password(description="2FA password:", placeholder="Only if Telegram asks")
password_button = widgets.Button(description="Verify 2FA", button_style="success")
status = widgets.HTML(value="<b>Ready.</b> Enter your Telegram API ID, API Hash and phone number.")
session_box = widgets.Textarea(description="Session:", layout=widgets.Layout(width="100%", height="110px"))
session_box.disabled = True
client = None
phone_code_hash = None

display(api_id_box, api_hash_box, phone_box, start_button, code_box, verify_button,
        password_box, password_button, status, session_box)
code_box.layout.display = "none"
verify_button.layout.display = "none"
password_box.layout.display = "none"
password_button.layout.display = "none"
session_box.layout.display = "none"

async def send_code(_):
    global client, phone_code_hash
    start_button.disabled = True
    try:
        api_id = int(api_id_box.value.strip())
        api_hash = api_hash_box.value.strip()
        phone = phone_box.value.strip()
        if not api_hash or not phone:
            raise ValueError("API Hash and phone number are required.")
        client = TelegramClient(StringSession(), api_id, api_hash)
        await client.connect()
        sent = await client.send_code_request(phone)
        phone_code_hash = sent.phone_code_hash
        code_box.layout.display = "flex"
        verify_button.layout.display = "inline-flex"
        status.value = "<b>Login code sent.</b> Enter it below."
    except Exception as exc:
        status.value = "<span style='color:#c00'>Could not send code: " + type(exc).__name__ + "</span>"
        if client:
            await client.disconnect()
            client = None
    finally:
        start_button.disabled = False

async def verify_code(_):
    verify_button.disabled = True
    try:
        await client.sign_in(phone=phone_box.value.strip(), code=code_box.value.strip(),
                             phone_code_hash=phone_code_hash)
        await show_session()
    except SessionPasswordNeededError:
        password_box.layout.display = "flex"
        password_button.layout.display = "inline-flex"
        status.value = "<b>Telegram 2-step verification is enabled.</b> Enter its password below."
    except Exception as exc:
        status.value = "<span style='color:#c00'>Verification failed: " + type(exc).__name__ + "</span>"
    finally:
        verify_button.disabled = False

async def verify_password(_):
    password_button.disabled = True
    try:
        await client.sign_in(password=password_box.value)
        await show_session()
    except Exception as exc:
        status.value = "<span style='color:#c00'>2FA verification failed: " + type(exc).__name__ + "</span>"
    finally:
        password_button.disabled = False

async def show_session():
    session_box.value = client.session.save()
    session_box.disabled = False
    session_box.layout.display = "flex"
    status.value = "<b>Login successful.</b> Copy the Session value directly into GitHub Actions secret TELEGRAM_SESSION_STRING. Never share it in chat or save it in the repository."
    await client.disconnect()

def on_start(_):
    asyncio.create_task(send_code(_))
def on_verify(_):
    asyncio.create_task(verify_code(_))
def on_password(_):
    asyncio.create_task(verify_password(_))

start_button.on_click(on_start)
verify_button.on_click(on_verify)
password_button.on_click(on_password)
